In [1]:
!pip install pandas numpy matplotlib scikit-learn

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

# 1. Merge Relational Databases

In [3]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path('../data')

# Already done by teammate
df_customers = pd.read_csv(DATA_DIR / 'olist_customers_dataset.csv')
df_orders = pd.read_csv(DATA_DIR / 'olist_orders_dataset.csv')
df = pd.merge(df_customers, df_orders, on='customer_id', how='left')

# Continue the chain
df_items = pd.read_csv(DATA_DIR / 'olist_order_items_dataset.csv')
df = pd.merge(df, df_items, on='order_id', how='left')

df_payments = pd.read_csv(DATA_DIR / 'olist_order_payments_dataset.csv')
df = pd.merge(df, df_payments, on='order_id', how='left')

df_reviews = pd.read_csv(DATA_DIR / 'olist_order_reviews_dataset.csv')
df = pd.merge(df, df_reviews, on='order_id', how='left')

df_products = pd.read_csv(DATA_DIR / 'olist_products_dataset.csv')
df = pd.merge(df, df_products, on='product_id', how='left')

df_sellers = pd.read_csv(DATA_DIR / 'olist_sellers_dataset.csv')
df = pd.merge(df, df_sellers, on='seller_id', how='left')

df_translation = pd.read_csv(DATA_DIR / 'product_category_name_translation.csv')
df = pd.merge(df, df_translation, on='product_category_name', how='left')

# Save master table
df.to_csv(DATA_DIR / 'olist_master.csv', index=False)

# 2. Data Preprocessing

In [4]:
print("Rows:    ", df.shape[0])
print("Columns: ", df.shape[1])

print("Data types of dataset:")
print(df.dtypes)

print("Missing values of dataset:")
print(df.isnull().sum()[df.isnull().sum() > 0])

df.head()

Rows:     119143
Columns:  40
Data types of dataset:
customer_id                       object
customer_unique_id                object
customer_zip_code_prefix           int64
customer_city                     object
customer_state                    object
order_id                          object
order_status                      object
order_purchase_timestamp          object
order_approved_at                 object
order_delivered_carrier_date      object
order_delivered_customer_date     object
order_estimated_delivery_date     object
order_item_id                    float64
product_id                        object
seller_id                         object
shipping_limit_date               object
price                            float64
freight_value                    float64
payment_sequential               float64
payment_type                      object
payment_installments             float64
payment_value                    float64
review_id                         object
revi

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,...,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,seller_zip_code_prefix,seller_city,seller_state,product_category_name_english
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,00e7ee1b050b8499577073aeb2a297a1,delivered,2017-05-16 15:05:35,2017-05-16 15:22:12,2017-05-23 10:47:57,...,1141.0,1.0,8683.0,54.0,64.0,31.0,8577.0,itaquaquecetuba,SP,office_furniture
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,29150127e6685892b6eab3eec79f59c7,delivered,2018-01-12 20:48:24,2018-01-12 20:58:32,2018-01-15 17:14:59,...,1002.0,3.0,10150.0,89.0,15.0,40.0,88303.0,itajai,SC,housewares
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,b2059ed67ce144a36e2aa97d2c9e9ad2,delivered,2018-05-19 16:07:45,2018-05-20 16:19:10,2018-06-11 14:31:00,...,955.0,1.0,8267.0,52.0,52.0,17.0,8577.0,itaquaquecetuba,SP,office_furniture
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,951670f92359f4fe4a63112aa7306eba,delivered,2018-03-13 16:06:38,2018-03-13 17:29:19,2018-03-27 23:22:42,...,1066.0,1.0,12160.0,56.0,51.0,28.0,8577.0,itaquaquecetuba,SP,office_furniture
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,6b7d50bd145f6fc7f33cebabd7e49d0f,delivered,2018-07-29 09:51:30,2018-07-29 10:10:09,2018-07-30 15:16:00,...,407.0,1.0,5200.0,45.0,15.0,35.0,14940.0,ibitinga,SP,home_confort


# 3. Remove not useful info / fill in values

In [5]:
# Drop columns
df = df.drop(columns=['review_comment_title', 'review_comment_message'], errors='ignore')
# Fill date columns
date_cols = ['order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date']
for col in date_cols:
    df[col] = df[col].fillna('not_available')

# Fill product category
df['product_category_name'] = df['product_category_name'].fillna('unknown')
df['product_category_name_english'] = df['product_category_name_english'].fillna('unknown')

# Fill product dimensions with median
dim_cols = ['product_name_lenght', 'product_description_lenght', 'product_photos_qty',
            'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
for col in dim_cols:
    df[col] = df[col].fillna(df[col].median())

# Drop rows with missing payment/review/order item data
df = df.dropna(subset=['payment_type', 'review_score', 'order_item_id'])

In [6]:
# Drop unique identifiers
df = df.drop(columns=[
    'customer_id', 'customer_unique_id', 'order_id',
    'product_id', 'seller_id', 'review_id'
], errors='ignore')

# Drop high-cardinality date/timestamp columns
df = df.drop(columns=[
    'order_approved_at', 'order_delivered_carrier_date',
    'order_delivered_customer_date', 'shipping_limit_date',
    'review_creation_date', 'review_answer_timestamp'
], errors='ignore')

# Drop redundant Portuguese category column
df = df.drop(columns=['product_category_name'], errors='ignore')

# Encode categoricals
cat_cols = ['customer_city', 'customer_state', 'order_status', 
            'payment_type', 'seller_city', 'seller_state',
            'product_category_name_english']

le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

# Extract date features
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['order_estimated_delivery_date'] = pd.to_datetime(df['order_estimated_delivery_date'], errors='coerce')

df['purchase_month'] = df['order_purchase_timestamp'].dt.month
df['purchase_dayofweek'] = df['order_purchase_timestamp'].dt.dayofweek
df['estimated_delivery_days'] = (
    df['order_estimated_delivery_date'] - df['order_purchase_timestamp']
).dt.days

df = df.drop(columns=['order_purchase_timestamp', 'order_estimated_delivery_date'])

print(df.isnull().sum()[df.isnull().sum() > 0])
print(df.shape)
print(df.dtypes)

Series([], dtype: int64)
(117329, 26)
customer_zip_code_prefix           int64
customer_city                      int64
customer_state                     int64
order_status                       int64
order_item_id                    float64
price                            float64
freight_value                    float64
payment_sequential               float64
payment_type                       int64
payment_installments             float64
payment_value                    float64
review_score                     float64
product_name_lenght              float64
product_description_lenght       float64
product_photos_qty               float64
product_weight_g                 float64
product_length_cm                float64
product_height_cm                float64
product_width_cm                 float64
seller_zip_code_prefix           float64
seller_city                        int64
seller_state                       int64
product_category_name_english      int64
purchase_month     

# 4. RFM feature engineering

Recency = how recently did they last buy?

Frequency = how many orders have they made?

Monetary = how much have they spent total?

In [7]:
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
reference_date = df['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

rfm = df.groupby('customer_unique_id').agg(
    recency=('order_purchase_timestamp', lambda x: (reference_date - x.max()).days),
    frequency=('order_id', 'nunique'),
    monetary=('payment_value', 'sum')
).reset_index()

print(rfm.head())
print(rfm.describe())

# 1 = churned (no purchase in last 180 days), 0 = active
rfm['churn'] = (rfm['recency'] > 180).astype(int)

print(rfm['churn'].value_counts())
print(rfm['churn'].value_counts(normalize=True))

KeyError: 'order_purchase_timestamp'

# 5. PCA

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['recency', 'frequency', 'monetary']])

# PCA
pca = PCA()
pca.fit(rfm_scaled)

print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Cumulative variance:", pca.explained_variance_ratio_.cumsum())

Explained variance ratio: [0.36254469 0.3328847  0.30457061]
Cumulative variance: [0.36254469 0.69542939 1.        ]


# 6. Save Final Results + Findings
Merged all 9 relational tables into olist_master.csv of 119,143 rows and 40 columns

Dropped review_comment_title (88% missing) and review_comment_message (58% missing) 

Filled missing product dimensions with median values, missing categories with 'unknown', and missing dates with 'not_available'

Dropped rows with missing payment, review, and order item data

Encoded 7 categorical columns (city, state, order status, payment type, etc.) into numeric labels for model compatibility

Engineered RFM features across 94,720 unique customers. Most customers bought only once (median frequency = 1) and haven't purchased in ~224 days on average

Defined churn as no purchase in the last 180 days. ~60% of customers are churned, ~40% are active

PCA was attempted but all 3 components explained variance nearly equally (36/33/30%), so raw RFM features were retained instead

In [ ]:
df.to_csv(DATA_DIR / 'olist_cleaned.csv', index=False)
rfm.to_csv(DATA_DIR / 'olist_rfm.csv', index=False)